In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2


from pathlib import Path
import csv
from typing import Any, Literal

import numpy as np
import spikeinterface as si
import spikeinterface.extractors as se
from spikeinterface.postprocessing.localization_tools import compute_center_of_mass
from matplotlib.figure import Figure
from numpy.typing import NDArray
from probeinterface import Probe

from Utils.si_utils import compute_template_ptp_summary, validate_data
from Utils.probe_plotting import ProbePlot

type FloatArray = NDArray[np.floating[Any]]
type IntArray = NDArray[np.integer[Any]]
type StructuredArray = NDArray[np.void]
type AxesArray = NDArray[np.object_]


In [ ]:
base_dir: Path = Path('/mnt/senzailab/Kai/#Recording/m14/')
date: str = "260615"
sessionID: str = "3"
session_dir = base_dir / date / f"{date}_{sessionID}"

probe_name = 'A'
only_good_units: bool = False
raw_unit_ids: list[int] | None = None  # None uses all units retained by only_good_units.
pre_spike: float = 0.5  # Average-window duration before each spike alignment.
post_spike: float = 1.5  # Average-window duration after each spike alignment.

save_figures: bool = False
export_unit_locations: bool = False
output_dir: Path = Path.home() / 'Developer/rfmapping/data/spikeinterface_outputs' / f'{session_dir.name}_Probe{probe_name}'

kilosort_run: str = f'kilosort_{sessionID}'
kilosort_dir: Path = session_dir / 'kilosort' / f'Probe{probe_name}' / kilosort_run

is_debug: bool = False

n_jobs: int = 32

raw_num_channels: int = 384
raw_dtype: Literal['int16'] = 'int16'
raw_sampling_frequency: float = 30_000.0
max_spikes_per_unit: int = 2_000

gain_to_uv: float = 0.195
offset_to_uv: float = 0.0
local_channel_mode: Literal['same_shank', 'same_x_column'] = 'same_shank'  # same_x_column: fixed x, nearest along y.
local_channel_count: int = 5  # Display only: best + four nearest channels in the selected mode.
unit_location_method: Literal['center_of_mass'] = 'center_of_mass'
unit_location_feature: Literal['ptp'] = 'ptp'  # Contact weight = raw mean-template PTP.
unit_location_radius_um: float = 75.0
probe_ptp_scale: Literal['per_unit', 'global_uv'] = 'per_unit'  # Display only.
probe_contact_radius_um: float = 6.0  # Display only: fixed 12 µm-diameter circles.
waveform_seed: int = 0  # SpikeInterface uniform-selection seed; no global np.random.seed.
heatmap_baseline_end_ms: float = -pre_spike / 2.0  # Display only: first half of the pre-spike window.
overwrite_waveform_analyzer: bool = False  # Reuse the cache; mismatched extensions are still refreshed.

raw_file: Path = (
        session_dir
        / session_dir.parent.name
        / 'Record Node 102/experiment1/recording1/continuous'
        / f'OneBox-104.Probe{probe_name}'
        / 'continuous.dat'
)
waveform_analyzer_folder: Path = (
        Path.home()
        / 'Developer/rfmapping/data/spikeinterface_analyzers'
        / f'{session_dir.name}_Probe{probe_name}_raw_uniform_{max_spikes_per_unit}_seed{waveform_seed}_dense'
)

_, result = validate_data(recording_file=raw_file, kilosort_dir=kilosort_dir, save_figures=save_figures,
                          export_unit_locations=export_unit_locations, output_dir=output_dir)

if _ and is_debug:
    print(f'Kilosort directory: {result["kilosort_directory"]}')
    print(f'Raw recording: {result["raw_recording"]}')
    print(f"Output directory: {output_dir}")
    print('\n')

    print(f"{'file':<28} {'shape':<20} dtype")
    print("-" * 62)

    for name, info in result["files"].items():
        print(f"{name:<28} {str(info['shape']):<20} {info['dtype']}")


In [ ]:
sorting: si.BaseSorting = se.read_kilosort(kilosort_dir, keep_good_only=only_good_units)

if raw_unit_ids is not None:
    present_unit_ids: set[int] = {int(unit_id) for unit_id in sorting.unit_ids}
    missing_unit_ids: list[int] = sorted(set(raw_unit_ids) - present_unit_ids)
    if missing_unit_ids:
        raise ValueError(f'raw_unit_ids not present in the sorting: {missing_unit_ids}')
    sorting = sorting.select_units(raw_unit_ids)

if sorting.get_num_units() == 0:
    raise ValueError('No units remain after filtering.')

unit_ids: list[int] = sorting.unit_ids.tolist()
spike_counts: dict[int, int] = {
    int(unit_id): int(count)
    for unit_id, count in sorting.count_num_spikes_per_unit().items()
}

quality_by_unit: dict[int, str] = {
    unit_id: str(sorting.get_property("KSLabel")[unit_index])
    for unit_index, unit_id in enumerate(unit_ids)
}

channel_map: IntArray = np.load(kilosort_dir / 'channel_map.npy')
channel_positions: FloatArray = np.load(kilosort_dir / 'channel_positions.npy')
if channel_map.shape != (raw_num_channels,) or channel_positions.shape != (raw_num_channels, 2):
    raise ValueError('raw_num_channels, channel_map.npy, and channel_positions.npy do not agree.')

recording: si.BaseRecording = si.read_binary(
    file_paths=raw_file,
    sampling_frequency=raw_sampling_frequency,
    dtype=raw_dtype,
    num_channels=raw_num_channels,
    gain_to_uV=gain_to_uv,
    offset_to_uV=offset_to_uv,
    is_filtered=False,
)
recording = recording.select_channels(channel_map)
probe: Probe = Probe(ndim=2, si_units='um')
probe.set_contacts(
    channel_positions,
    shapes='circle',
    shape_params={'radius': probe_contact_radius_um},
)
probe.set_device_channel_indices(np.arange(raw_num_channels))
recording = recording.set_probe(probe)
plotting_probe: Probe = recording.get_probe()

if (spike_times := np.load(kilosort_dir / 'spike_times.npy', mmap_mode='r')).max() >= recording.get_num_frames():
    raise ValueError('The raw file is shorter than the largest Kilosort spike time.')
if recording.get_num_segments() != 1:
    raise ValueError('This notebook currently expects one recording segment.')

recording_duration_minutes: float = recording.get_total_duration() / 60.0
if is_debug:
    print(f'Units selected: {len(unit_ids)} · total spikes: {sum(spike_counts.values()):,}')
    print(f'Recording: {recording_duration_minutes:.2f} min · channels: {recording.get_num_channels()}')
    print(f'Units used for analysis and plotting: {len(unit_ids)}')

## Build the SpikeInterface waveform analyzer

The analyzer is dense so every probe contact is computed from raw data. `random_spikes(method='uniform')` samples at most 2,000 events per unit over the complete spike train, and `templates` uses SpikeInterface's streaming accumulator to calculate the mean without creating a `waveforms` extension.

In [ ]:
expected_selection_params: dict[str, object] = {
    'method': 'uniform',
    'max_spikes_per_unit': max_spikes_per_unit,
    'margin_size': None,
    'seed': waveform_seed,
}

waveform_analyzer: si.SortingAnalyzer
if waveform_analyzer_folder.exists() and not overwrite_waveform_analyzer:
    waveform_analyzer = si.load_sorting_analyzer(waveform_analyzer_folder)
    if not np.array_equal(waveform_analyzer.unit_ids, sorting.unit_ids):
        raise ValueError('Cached analyzer unit IDs do not match the current sorting.')
    if not np.array_equal(waveform_analyzer.channel_ids, recording.channel_ids):
        raise ValueError('Cached analyzer channel IDs do not match the current recording.')
    if waveform_analyzer.sparsity is not None:
        raise ValueError('Cached analyzer is sparse; rebuild it with overwrite_waveform_analyzer = True.')
    print(f'Loaded waveform analyzer: {waveform_analyzer_folder}')
else:
    waveform_analyzer_folder.parent.mkdir(parents=True, exist_ok=True)
    waveform_analyzer = si.create_sorting_analyzer(
        sorting,
        recording,
        format='binary_folder',
        folder=waveform_analyzer_folder,
        sparse=False,
        overwrite=overwrite_waveform_analyzer,
        n_jobs=n_jobs,
        chunk_duration='1s',
        progress_bar=True,
    )

selection_extension: si.ComputeRandomSpikes | None = waveform_analyzer.get_extension('random_spikes')
if not isinstance(selection_extension,
                  si.ComputeRandomSpikes) or selection_extension.params != expected_selection_params:
    waveform_analyzer.compute('random_spikes', **expected_selection_params)
    selection_extension = waveform_analyzer.get_extension('random_spikes')
if not isinstance(selection_extension, si.ComputeRandomSpikes):
    raise TypeError('Unexpected random_spikes extension type.')
if waveform_analyzer.has_extension('waveforms'):
    raise ValueError('This analyzer must not store individual waveforms.')

template_extension: si.ComputeTemplates | None = waveform_analyzer.get_extension('templates')
template_params_match: bool = (
        isinstance(template_extension, si.ComputeTemplates)
        and template_extension.params.get('operators') == ['average']
        and template_extension.params.get('ms_before') == pre_spike
        and template_extension.params.get('ms_after') == post_spike
)
if not template_params_match:
    waveform_analyzer.compute(
        'templates',
        ms_before=pre_spike,
        ms_after=post_spike,
        operators=['average'],
        n_jobs=n_jobs,
        chunk_duration='1s',
        progress_bar=True,
    )
    template_extension = waveform_analyzer.get_extension('templates')
if not isinstance(template_extension, si.ComputeTemplates):
    raise TypeError('Unexpected templates extension type.')
if waveform_analyzer.has_extension('unit_locations'):
    waveform_analyzer.delete_extension('unit_locations')

selected_spikes: StructuredArray = selection_extension.get_random_spikes()
selected_counts_by_index: IntArray = np.bincount(
    selected_spikes['unit_index'],
    minlength=sorting.get_num_units(),
)
selected_spike_counts: dict[int, int] = {
    unit_id: int(selected_counts_by_index[unit_index])
    for unit_index, unit_id in enumerate(unit_ids)
}
expected_selected_counts: dict[int, int] = {
    unit_id: min(spike_counts[unit_id], max_spikes_per_unit)
    for unit_id in unit_ids
}
count_mismatches: dict[int, tuple[int, int]] = {
    unit_id: (selected_spike_counts[unit_id], expected_selected_counts[unit_id])
    for unit_id in unit_ids
    if selected_spike_counts[unit_id] != expected_selected_counts[unit_id]
}
if count_mismatches:
    raise ValueError(f'Unexpected random-spike counts: {count_mismatches}')

small_unit_count: int = sum(count < max_spikes_per_unit for count in spike_counts.values())
print(f'Selected {len(selected_spikes):,} spikes across {len(unit_ids)} units.')
print(f'{small_unit_count} units have fewer than {max_spikes_per_unit:,} spikes and therefore use every spike.')
print(f'Extensions: {waveform_analyzer.get_loaded_extension_names()}')


probe_plot = ProbePlot(
    plotting_probe,
    session_dir,
    unit_ids=unit_ids,
    waveform_analyzer=waveform_analyzer,
    template_extension=template_extension,
    probe_ptp_scale=probe_ptp_scale,
    is_debug=is_debug,
    probe_name=probe_name,
    save_figures=save_figures,
    output_dir=output_dir,
)


## Random waveform spikes across the full recording

Gray ticks are all spikes for each selected unit; colored ticks are the events used by SpikeInterface for its average. The positions are exact recording times with no jitter. Sampling is uniform over existing spike events, so silent periods remain empty and burstier periods naturally contain more selected spikes.

In [ ]:
print(f"{'unit':>6} {'selected':>10} {'total':>10} {'time coverage':>15}")
print('-' * 47)
for unit_number, unit_id in enumerate(unit_ids):
    unit_index: int = int(sorting.id_to_index(unit_id))
    all_spike_samples: IntArray = sorting.get_unit_spike_train(unit_id=unit_id)
    all_spike_minutes: FloatArray = all_spike_samples / raw_sampling_frequency / 60.0
    selected_unit_mask: NDArray[np.bool_] = selected_spikes['unit_index'] == unit_index
    selected_spike_minutes: FloatArray = selected_spikes['sample_index'][selected_unit_mask] / raw_sampling_frequency / 60.0
    time_coverage_percent: float = (
        float(np.ptp(selected_spike_minutes)) / recording_duration_minutes * 100.0
        if len(selected_spike_minutes) > 1
        else 0.0
    )
    print(
        f'{unit_id:>6} {selected_spike_counts[unit_id]:>10,} '
        f'{spike_counts[unit_id]:>10,} {time_coverage_percent:14.1f}%'
    )

probe_plot.plot_waveform_spike_selection_times(
    sorting=sorting,
    selected_spikes=selected_spikes,
    raw_sampling_frequency=raw_sampling_frequency,
    recording_duration_minutes=recording_duration_minutes,
    selected_spike_counts=selected_spike_counts,
    spike_counts=spike_counts,
)


## SpikeInterface local average waveforms

Each panel is a simple five-row stack: the maximum-PTP channel plus four neighbors selected by `local_channel_mode`. Use `same_shank` for the nearest contacts within the best channel's Kilosort `kcoords` shank, or `same_x_column` for the nearest contacts with exactly the same x coordinate (therefore nearest along y). Circular row markers identify channels, with the best channel filled. For visibility only, each row is shifted by its mean over the first half of the pre-spike window; this removes a constant DC offset but performs no smoothing or interpolation. PTP, best-channel traces, and unit locations continue to use the unshifted averages.

In [ ]:
template_array: FloatArray = template_extension.get_templates(
    unit_ids=unit_ids,
    operator='average',
)
time_ms: FloatArray = (
                              np.arange(template_array.shape[1]) - template_extension.nbefore
                      ) / waveform_analyzer.sampling_frequency * 1_000.0
heatmap_baseline_mask: NDArray[np.bool_] = time_ms <= heatmap_baseline_end_ms
if not np.any(heatmap_baseline_mask):
    raise ValueError('The heatmap baseline window contains no waveform samples.')
heatmap_baseline_uv: FloatArray = np.mean(
    template_array[:, heatmap_baseline_mask, :],
    axis=1,
    keepdims=True,
)
heatmap_template_array: FloatArray = template_array - heatmap_baseline_uv
channel_locations: FloatArray = waveform_analyzer.get_channel_locations()
analyzer_contact_positions: FloatArray = waveform_analyzer.get_probe().contact_positions
plotting_contact_positions: FloatArray = plotting_probe.contact_positions
if not np.allclose(channel_locations, analyzer_contact_positions):
    raise ValueError('Analyzer channel order and probe contact order do not match.')
if not np.allclose(channel_locations, plotting_contact_positions):
    raise ValueError('Plotting probe contact order does not match analyzer channels.')

template_ptp_summary = compute_template_ptp_summary(template_array)
best_channel_indices: IntArray = template_ptp_summary.best_channel_indices
best_channel_ptp_uv: FloatArray = template_ptp_summary.max_ptp_by_unit
time_step_ms: float = float(np.median(np.diff(time_ms)))
time_edges_ms: FloatArray = np.r_[
    time_ms[0] - time_step_ms / 2.0,
    (time_ms[:-1] + time_ms[1:]) / 2.0,
    time_ms[-1] + time_step_ms / 2.0,
]
template_limit_uv: float = max(float(np.max(np.abs(heatmap_template_array))), np.finfo(float).eps)

channel_ids_array: IntArray = np.asarray(waveform_analyzer.channel_ids)
expected_channel_count: int = waveform_analyzer.get_num_channels()
if template_array.shape[2] != expected_channel_count:
    raise ValueError('Template columns do not match the analyzer channel count.')
if channel_locations.shape != (expected_channel_count, 2):
    raise ValueError('Channel locations do not match the analyzer channel count.')
if len(channel_ids_array) != expected_channel_count or len(np.unique(channel_ids_array)) != expected_channel_count:
    raise ValueError('Analyzer channel IDs are missing or duplicated.')

kilosort_ops: dict[str, Any] = np.load(
    kilosort_dir / 'ops.npy',
    allow_pickle=True,
).item()
probe_ops_value: Any = kilosort_ops.get('probe')
if not isinstance(probe_ops_value, dict):
    raise ValueError("Kilosort ops.npy does not contain a 'probe' mapping.")
probe_ops: dict[str, Any] = probe_ops_value
if 'chanMap' not in probe_ops or 'kcoords' not in probe_ops:
    raise ValueError("Kilosort probe metadata must contain 'chanMap' and 'kcoords'.")
ops_channel_map: IntArray = np.asarray(probe_ops['chanMap']).squeeze().astype(int)
channel_shank_ids: IntArray = np.asarray(probe_ops['kcoords']).squeeze().astype(int)
if ops_channel_map.shape != channel_map.shape or not np.array_equal(ops_channel_map, channel_map):
    raise ValueError('Kilosort kcoords order does not align with channel_map.npy.')
if channel_shank_ids.shape != (expected_channel_count,):
    raise ValueError('Kilosort kcoords do not align with analyzer channels.')
if local_channel_mode not in ('same_shank', 'same_x_column'):
    raise ValueError("local_channel_mode must be 'same_shank' or 'same_x_column'.")
if local_channel_count < 1:
    raise ValueError('local_channel_count must be positive.')
local_channel_axis_label: str = (
    'Same-shank channels'
    if local_channel_mode == 'same_shank'
    else 'Same-x-column channels'
)
local_channel_mode_description: str = (
    'same Kilosort shank'
    if local_channel_mode == 'same_shank'
    else 'same x column, nearest along y'
)


probe_plot.plot_local_average_heatmaps(
    heatmap_template_array=heatmap_template_array,
    best_channel_indices=best_channel_indices,
    channel_locations=channel_locations,
    channel_shank_ids=channel_shank_ids,
    local_channel_mode=local_channel_mode,
    local_channel_count=local_channel_count,
    time_edges_ms=time_edges_ms,
    template_limit_uv=template_limit_uv,
    channel_ids_array=channel_ids_array,
    local_channel_axis_label=local_channel_axis_label,
    local_channel_mode_description=local_channel_mode_description,
)


## Best-channel SpikeInterface average waveforms

The maximum-PTP contact is selected from the newly calculated raw average. Every circle is one stored time sample; connecting lines are only straight segments between adjacent samples.

In [ ]:
best_channel_waveforms: FloatArray = np.stack([
    template_array[unit_index, :, int(best_channel_indices[unit_index])]
    for unit_index in range(len(unit_ids))
])

print(f"{'unit':>6} {'used/total':>19} {'channel':>8} {'x_um':>9} {'y_um':>9} {'ptp_uv':>10}")
print('-' * 68)
for unit_number, unit_id in enumerate(unit_ids):
    channel_index: int = int(best_channel_indices[unit_number])
    waveform_uv: FloatArray = best_channel_waveforms[unit_number]
    channel_id: int = int(waveform_analyzer.channel_ids[channel_index])
    x_um: float = float(channel_locations[channel_index, 0])
    y_um: float = float(channel_locations[channel_index, 1])
    ptp_uv: float = float(best_channel_ptp_uv[unit_number])
    used_total: str = f'{selected_spike_counts[unit_id]:,}/{spike_counts[unit_id]:,}'
    print(f'{unit_id:>6} {used_total:>19} {channel_id:>8} {x_um:9.1f} {y_um:9.1f} {ptp_uv:10.2f}')

probe_plot.plot_best_channel_averages(
    best_channel_waveforms=best_channel_waveforms,
    best_channel_indices=best_channel_indices,
    best_channel_ptp_uv=best_channel_ptp_uv,
    channel_locations=channel_locations,
    time_ms=time_ms,
)


## SpikeInterface PTP gradient or maximum-contact circle

`plot_probe_ptp(...)` reuses one raw-average PTP calculation for two views. With `plot_the_center=False`, every contact shows the PTP gradient. With `plot_the_center=True`, the gradient is replaced by one circular marker at the maximum-PTP contact. Here `center` deliberately means the highest-weight contact requested in this plot, not the separate center-of-mass estimate below. PTP normalization remains display-only.

In [ ]:
ptp_gradient_figure: Figure
ptp_gradient_axes: AxesArray
ptp_gradient_figure, ptp_gradient_axes = probe_plot.plot_probe_ptp(
    unit_ids_to_show=unit_ids,
    plot_the_center=False,
    show_other_units=False,
    is_in_one_plot=False,
)
probe_plot.finalize_figure(ptp_gradient_figure, 'spikeinterface_ptp_probe_maps.png')

## Unit positions from SpikeInterface raw averages

`center_of_mass` uses each unit's newly calculated raw-template PTP and probe coordinates. A SpikeInterface `ChannelSparsity` is centered on the raw PTP-best channel within 75 µm before calling `compute_center_of_mass`; this prevents unfiltered channel DC offsets from choosing the neighborhood. Kilosort template amplitudes are not involved.

In [ ]:
if unit_location_method != 'center_of_mass':
    raise ValueError('This notebook currently implements center_of_mass unit locations.')
location_sparsity_mask: NDArray[np.bool_] = np.zeros(
    (sorting.get_num_units(), waveform_analyzer.get_num_channels()),
    dtype=bool,
)
for unit_index in range(sorting.get_num_units()):
    best_channel_index: int = int(best_channel_indices[unit_index])
    distances: FloatArray = np.linalg.norm(
        channel_locations - channel_locations[best_channel_index],
        axis=1,
    )
    location_sparsity_mask[unit_index] = distances <= unit_location_radius_um

location_sparsity: si.ChannelSparsity = si.ChannelSparsity(
    mask=location_sparsity_mask,
    unit_ids=sorting.unit_ids,
    channel_ids=waveform_analyzer.channel_ids,
)
dense_location_templates: si.Templates = si.Templates(
    templates_array=template_array,
    sampling_frequency=waveform_analyzer.sampling_frequency,
    nbefore=template_extension.nbefore,
    is_in_uV=True,
    channel_ids=waveform_analyzer.channel_ids,
    unit_ids=sorting.unit_ids,
    probe=waveform_analyzer.get_probe(),
)
sparse_location_templates: si.Templates = dense_location_templates.to_sparse(location_sparsity)
unit_locations: FloatArray = compute_center_of_mass(
    sparse_location_templates,
    feature=unit_location_feature,
)
best_contact_locations: FloatArray = channel_locations[best_channel_indices]
unit_location_offsets_um: FloatArray = np.linalg.norm(unit_locations - best_contact_locations, axis=1)
if np.any(unit_location_offsets_um > unit_location_radius_um + 1e-6):
    raise ValueError('A unit center fell outside its raw PTP-defined location radius.')
location_order: IntArray = np.lexsort((unit_locations[:, 0], unit_locations[:, 1]))
"""
--------- Print Cluster Detail ------
print(f"{'unit':>6} {'quality':>10} {'x_um':>10} {'y_um':>10}")
print('-' * 42)
for ordered_index in location_order[:20]:
    unit_index: int = int(ordered_index)
    listed_unit_id: int = unit_ids[unit_index]
    quality: str = quality_by_unit[listed_unit_id]
    x_um: float = float(unit_locations[unit_index, 0])
    y_um: float = float(unit_locations[unit_index, 1])
    print(f'{listed_unit_id:>6} {quality:>10} {x_um:10.1f} {y_um:10.1f}')"""


probe_plot.plot_unit_locations(
    sorting=sorting,
    unit_locations=unit_locations,
)
if export_unit_locations:
    with (output_dir / 'spikeinterface_unit_locations.csv').open('w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['unit_id', 'quality', 'x_um', 'y_um'])
        for ordered_index in location_order:
            unit_index = int(ordered_index)
            unit_id = unit_ids[unit_index]
            writer.writerow([
                unit_id,
                quality_by_unit[unit_id],
                float(unit_locations[unit_index, 0]),
                float(unit_locations[unit_index, 1]),
            ])


## Maximum-PTP contact · one circle per unit

This reuses `plot_probe_ptp(...)` with `plot_the_center=True`. Each requested unit is represented by exactly one circular marker at the contact with the largest raw-average PTP. `is_in_one_plot=True` combines the requested units on one probe; `False` gives each unit its own panel. `show_other_units=True` adds only faint maximum-contact circles, never per-spike clouds.

In [ ]:
max_contact_figure: Figure
max_contact_axes: AxesArray
max_contact_figure, max_contact_axes = probe_plot.plot_probe_ptp(
    unit_ids_to_show=unit_ids,
    plot_the_center=True,
    show_other_units=False,
    is_in_one_plot=False,
)
probe_plot.finalize_figure(max_contact_figure, 'spikeinterface_max_ptp_contacts.png')

## Verification summary

The final checks make the calculation provenance explicit and fail if a Kilosort template dependency or stored individual-waveform extension is reintroduced.

In [ ]:
if waveform_analyzer.has_extension('waveforms'):
    raise AssertionError('Individual waveforms were stored unexpectedly.')
if selection_extension.params != expected_selection_params:
    raise AssertionError('Random-spike parameters changed unexpectedly.')

print('Waveform source: continuous.dat via SpikeInterface streaming templates')
print(f'Selection: uniform without replacement · min(total, {max_spikes_per_unit:,}) per unit')
print(f'Seed: {waveform_seed} · global NumPy random state is not used')
print(
    f'Window: {pre_spike:.1f} ms before + {post_spike:.1f} ms after · '
    f'{template_extension.nbefore} + {template_array.shape[1] - template_extension.nbefore} samples'
)
print(f'Analyzer: dense {waveform_analyzer.get_num_channels()}-channel averages · no waveforms extension')
print(
    f'Local heatmap: maximum + {local_channel_count - 1} nearest channels · '
    f'{local_channel_mode_description} '
    '· five stacked rows · circular channel markers'
)
print(f'Probe PTP display: {probe_ptp_scale} · raw PTP remains unchanged')
print(f'Probe contacts: fixed-radius circles · {2.0 * probe_contact_radius_um:.1f} µm diameter')
print('Probe position marker: one circle at the maximum raw-average PTP contact')
print(
    f'Unit location: SpikeInterface {unit_location_method} · {unit_location_feature} · '
    f'raw PTP-defined radius {unit_location_radius_um:.0f} µm'
)
print(f'Cache: {waveform_analyzer_folder}')